### Standalone evaluation transcription helper

This notebook requires an explicitly selected Whisper checkpoint
and a local copy of the NileTTS evaluation audio.

The original notebook did not record its checkpoint and has no saved outputs.
It cannot be confirmed as the source of the final ASR results CSV.

Selecting a checkpoint enables a new inference run; it does not establish
the provenance of the historical results.

In [ ]:
# =====
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "requirements-asr.txt").is_file():
        requirements_file = candidate / "requirements-asr.txt"
        break
else:
    raise FileNotFoundError("Could not find requirements-asr.txt")

%pip install -r "{requirements_file}"
# =====

In [ ]:
# =====
from pathlib import Path

import pandas as pd
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration
)

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (
        (candidate / "1_ASR").is_dir()
        and (candidate / "2_QA_GENERATION").is_dir()
    ):
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Open the notebook from inside the project folder."
    )

ASR_DIR = PROJECT_ROOT / "1_ASR"
DATASET_DIR = PROJECT_ROOT / "NileTTS-dataset"

# Set an absolute local checkpoint path or a Hugging Face model ID.
MODEL_PATH = None

if MODEL_PATH is None:
    raise ValueError(
        "Set MODEL_PATH to the Whisper checkpoint you intend to evaluate. "
        "The original checkpoint was not recorded."
    )

eval_csv = DATASET_DIR / "metadata_eval.csv"

if not eval_csv.is_file():
    raise FileNotFoundError(
        f"Missing evaluation metadata: {eval_csv}"
    )

eval_df = pd.read_csv(eval_csv, sep="|")

eval_df["audio_file"] = (
    eval_df["audio_file"]
    .astype(str)
    .str.replace("wav/", "wavs/", regex=False)
)

eval_df["audio_path"] = eval_df["audio_file"].apply(
    lambda name: str(DATASET_DIR / name)
)

missing_audio = eval_df[
    ~eval_df["audio_path"].apply(lambda path: Path(path).is_file())
]

if not missing_audio.empty:
    raise FileNotFoundError(
        f"Missing {len(missing_audio)} evaluation audio files. "
        "Check DATASET_DIR and complete the dataset download."
    )

processor = WhisperProcessor.from_pretrained(
    str(MODEL_PATH),
    language="Arabic",
    task="transcribe"
)

model = WhisperForConditionalGeneration.from_pretrained(
    str(MODEL_PATH)
)

print("Selected checkpoint:", MODEL_PATH)
print("Evaluation rows:", len(eval_df))
# =====

In [ ]:
import re
import time
import numpy as np
import pandas as pd
import torch
import librosa

from tqdm.auto import tqdm
from jiwer import wer, cer

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None


def normalize_arabic_for_eval(text):
    text = str(text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"ـ", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def transcribe_one_audio(audio_path):
    audio_array, sr = librosa.load(audio_path, sr=16000, mono=True)

    inputs = processor(
        audio_array,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            max_length=448
        )

    predicted_text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    return predicted_text


results = []
start = time.time()

# IMPORTANT: use eval_df here, not train_df
for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Generating eval ASR"):
    audio_path = row["audio_path"]
    reference_text = str(row["text"])

    try:
        predicted_text = transcribe_one_audio(audio_path)

        ref_norm = normalize_arabic_for_eval(reference_text)
        hyp_norm = normalize_arabic_for_eval(predicted_text)

        sample_wer = wer(ref_norm, hyp_norm)
        sample_cer = cer(ref_norm, hyp_norm)

    except Exception as e:
        predicted_text = ""
        sample_wer = 1.0
        sample_cer = 1.0
        print(f"Error at index {idx}: {e}")

    results.append({
        "audio_file": row["audio_file"],
        "speaker_name": row["speaker_name"],
        "reference_text": reference_text,
        "asr_text": predicted_text,
        "wer": sample_wer,
        "cer": sample_cer
    })


asr_eval_df = pd.DataFrame(results)

print("Eval ASR WER:", asr_eval_df["wer"].mean())
print("Eval ASR CER:", asr_eval_df["cer"].mean())

# =====
output_file = ASR_DIR / "niletts_asr_eval_results.csv"

# asr_eval_df.to_csv(
#     "niletts_asr_eval_results.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

asr_eval_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_file)
# =====

asr_eval_df.head()